In [ ]:
# Setup correct venv
#!pip install "plotly<6.0.0" ##---> workaround titleside error

In [ ]:
from sklearn.preprocessing import FunctionTransformer
from pathlib import Path
from joblib import load
import utils
import pandas as pd
from umap import UMAP

from gtda.mapper import (
    CubicalCover,
    make_mapper_pipeline,
    plot_static_mapper_graph,
    plot_interactive_mapper_graph
)

from sklearn.cluster import DBSCAN

Pancreas endocrinogenesis dataset [A. Bastidas-Ponce, S. Tritschler et al., 2019](https://github.com/theislab/pancreatic-endocrinogenesis/tree/master)

Pancreatic epithelial and Ngn3-Venus fusion (NVF) cells during secondary transition with transcriptome profiles sampled from embryonic day 15.5.

Endocrine cells are derived from endocrine progenitors located in the pancreatic epithelium. Endocrine commitment terminates in four major fates: glucagon- producing α-cells, insulin-producing β-cells, somatostatin-producing δ-cells and ghrelin-producing ε-cells. [Paper]( https://doi.org/10.1242/dev.173849
)

In [ ]:
# Navigate to the scripts folder and run `python endocrinogenesis.py`
root = Path.cwd().parent
cache_dir = utils._cache_file_folder(root, "data/Pancreas/cache")
hash_fname = "pipeline_bundle_5d6f526d04ee.joblib" ## enter the cached pipeline bundle stored in data/Pancreas
bundle = load(cache_dir / hash_fname)

embedding = bundle["embedding_collection"]["embedding"]
print("Shape of pancreas data UMAP: ", embedding.shape)

In [ ]:
hvg_genes = bundle["highly_variable_genes"]
hvg_genes = hvg_genes.index[hvg_genes].tolist()
len(hvg_genes)

In [ ]:
metadata = pd.DataFrame(bundle["adata_metadata"])
logNormData = pd.DataFrame(bundle["adata_collection"]["logNormal"].toarray(), index = metadata.index, columns = hvg_genes)
metadata = pd.concat([logNormData, metadata], axis = 1)
metadata

In [ ]:
metadata["clusters_coarse"] = metadata["clusters_coarse"].cat.codes.values
metadata["clusters"] = metadata["clusters"].cat.codes.values

Interactive and Static TDA

In [ ]:
# Creating mapper
cover = CubicalCover(n_intervals=10, overlap_frac=0.3)
clusterer = DBSCAN()
filter_func = FunctionTransformer(lambda x: x)

pipe = make_mapper_pipeline(
    filter_func=filter_func, 
    cover=cover,
    clusterer=clusterer,
    verbose=False,
    n_jobs=1,
)

In [ ]:
#MIP = MapperInteractivePlotter(pipe, data) buggy
color_data = metadata.drop(["clusters", "clusters_coarse"], inplace = False, axis = 1)
plot_interactive_mapper_graph(pipe, embedding, color_data=color_data)


In [ ]:
# Static TDA on the endocrinogenesis UMAP
fig = plot_static_mapper_graph(pipe, embedding)
fig.show(config={'scrollZoom': True})